[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/363172-risk-model-diversification.ipynb)

# Quantitative Investment Diversification Strategies via Various Risk Models — reproduction

**Maysam Khodayari Gharanchaei, Prabhu Prasad Panda & Xilin Chen** · [arXiv 2407.01550](https://arxiv.org/abs/2407.01550) (2024)

The paper runs a 3×3 grid of risk models — three covariance estimators (single-factor, constant-correlation, sample + shrinkage) × three long-only optimizers (min-variance, **max-diversification**, risk-parity) — on "1000 risky US stocks, CRSP, 1970–2023", monthly rebalanced. Headline: max-diversification + constant-correlation Sharpe **1.04** with **+41.8%** excess return; its best variant (max-div + shrinkage) **1.34**.

This notebook runs the **same logic as our engine** on **public data only** and lands on our published numbers. It pulls the Ken French **49 Industry Portfolios** (value-weighted monthly, dividend-inclusive) and the **F-F Research Data Factors** straight from the Ken French Data Library at run time (the paper's stock selection is undisclosed and survivorship-suspected; the industry panel is the free, survivorship-clean control). It then rebuilds all three covariance estimators, all three CVXPY optimizers, the 60-month rolling out-of-sample backtest (realised OOS 1975–2023, n=588 months), and prints **claimed → measured** with the multiple-testing + Deflated-Sharpe gates.

**What you'll see:** the headline measures **0.595** (t=4.16), not 1.04 — and the claimed +41.8%/yr excess measures **+9.6%/yr**. The result is statistically *real* (9/9 configs survive BHY, DSR=1.00) but ~2× overstated — and what remains is market beta, not alpha (FF3 α≈0, R²=0.95). Verdict: **Overstated**. The full C0–C7 gate suite (crisis windows, frictions, decay, spanning) lives in the [report](https://market-graphs.com/praxis/papers/363172). Runs in ~1–4 min (5,292 CVXPY solves).


In [ ]:
# Pinned to the engine's versions so the numbers reproduce exactly.
%pip install -q numpy==2.1.2 pandas==2.2.3 scipy==1.14.1 cvxpy==1.9.2 clarabel scikit-learn==1.8.0

In [ ]:
"""Praxis reproduction — paper 363172 (arXiv 2407.01550), "Quantitative
Investment Diversification Strategies via Various Risk Models" (Gharanchaei,
Panda & Chen, 2024).

Self-contained port of the engine blocks (data/kenfrench, blocks/covariance,
blocks/optimizers, blocks/backtest, blocks/metrics, gates/deflation) + the
exemplar wiring. Public data only, pulled at run time. Deterministic:
closed-form estimators + convex solves, no RNG.

Documented deviations from the paper (identical to the engine / published card):
  - CRSP 1000 stocks   -> Ken French 49 industry portfolios (free, survivorship-
                          clean; the paper's stock selection is UNDISCLOSED)
  - GUROBI             -> open CVXPY solvers (CLARABEL/SCS/OSQP; numerically
                          equivalent for these convex problems)
  - undisclosed cap    -> 0.10 (estimated)
  - undisclosed target -> Ledoit-Wolf shrinkage (estimated)
  - lookback           -> 60 months (paper: "past 60-month excess returns")

Window: 1970-2023 pull; realised OOS 1975-2023 (after the 60-month lookback),
n=588 months, 49 assets.
"""
import io
import re
import time
import urllib.request
import warnings
import zipfile
from dataclasses import dataclass

import cvxpy as cp
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm

warnings.filterwarnings("ignore")

PPY = 12
FOCAL = ("max_diversification", "constant_correlation")   # the paper's headline
BEST = ("max_diversification", "sample_shrinkage")        # the paper's best variant
CLAIMED = {FOCAL: {"sharpe": 1.04, "excess_return_pct": 41.8},
           BEST: {"sharpe": 1.34}}
LOOKBACK, CAP, COST_BPS = 60, 0.10, 10.0
START, END = "1970-01-01", "2023-12-31"

# ------------------------------------------------------------------ data -------
# Data is pulled directly from the Ken French Data Library at run time, for your
# own analysis, and is NOT redistributed with this repository.
KF_BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp"
MISSING = (-99.99, -999.0, -999.99)   # Ken French missing-data sentinels


def _fetch_zip_csv(zip_name):
    req = urllib.request.Request(f"{KF_BASE}/{zip_name}",
                                 headers={"User-Agent": "Mozilla/5.0 (praxis-repro)"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        buf = io.BytesIO(resp.read())
    with zipfile.ZipFile(buf) as zf:
        return zf.read(zf.namelist()[0]).decode("latin-1")


def _first_monthly_table(text):
    """First YYYYMM-indexed block of a Ken French CSV -> DataFrame of decimals.
    (49 industries: 'Average Value Weighted Returns -- Monthly'; factors file:
    the monthly Mkt-RF/SMB/HML/RF table.)"""
    lines = text.splitlines()
    start = next(i for i, ln in enumerate(lines) if re.match(r"^\s*\d{6},", ln))
    header = [c.strip() for c in lines[start - 1].split(",")]
    rows, idx = [], []
    for ln in lines[start:]:
        if not re.match(r"^\s*\d{6},", ln):
            break
        parts = ln.split(",")
        idx.append(parts[0].strip())
        rows.append([float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=header[1:])
    df.index = pd.PeriodIndex(idx, freq="M").to_timestamp(how="end").normalize()
    return df.mask(df.isin(MISSING)) / 100.0


def load_dataset(start=START, end=END):
    """{returns, rf, mkt_excess} aligned on the common month-end index."""
    ind = _first_monthly_table(_fetch_zip_csv("49_Industry_Portfolios_CSV.zip")).loc[start:end]
    ff = _first_monthly_table(_fetch_zip_csv("F-F_Research_Data_Factors_CSV.zip")).loc[start:end]
    idx = ind.index.intersection(ff.index)
    return {"returns": ind.loc[idx], "rf": ff.loc[idx, "RF"], "mkt_excess": ff.loc[idx, "Mkt-RF"]}


# ---------------------------------------- 3 covariance estimators --------------
def constant_correlation(X, m=None):
    """Elton-Gruber: every pair shares the average sample correlation."""
    X = np.asarray(X, dtype=float)
    s = X.std(axis=0, ddof=1)
    R = np.corrcoef(X, rowvar=False)
    n = R.shape[0]
    rho_bar = R[~np.eye(n, dtype=bool)].mean()
    Sigma = rho_bar * np.outer(s, s)
    np.fill_diagonal(Sigma, s ** 2)
    return Sigma


def single_factor(X, m):
    """Sharpe single-index: Sigma = beta beta' var_m + diag(residual var)."""
    X = np.asarray(X, dtype=float)
    m = np.asarray(m, dtype=float).ravel()
    T, n = X.shape
    var_m = m.var(ddof=1)
    mc = m - m.mean()
    beta = (X - X.mean(0)).T @ mc / (mc @ mc)
    resid = X - (X.mean(0) + np.outer(mc, beta))
    Sigma = np.outer(beta, beta) * var_m
    Sigma[np.diag_indices(n)] += resid.var(axis=0, ddof=1)
    return Sigma


def sample_shrinkage(X, m=None):
    """Ledoit-Wolf shrinkage toward scaled identity (paper's target undisclosed)."""
    from sklearn.covariance import LedoitWolf
    return LedoitWolf().fit(np.asarray(X, dtype=float)).covariance_


ESTIMATORS = {"single_factor": single_factor,
              "constant_correlation": constant_correlation,
              "sample_shrinkage": sample_shrinkage}


# ------------------------------------------- 3 optimizers (CVXPY) --------------
def _apply_cap(w, cap):
    """Deterministic water-filling: clip to [0, cap], redistribute pro-rata."""
    w = np.clip(w, 0, None)
    w = w / w.sum()
    if cap >= 1.0:
        return w
    k = int((w > 0).sum())
    if k * cap < 1.0 - 1e-9:
        return np.where(w > 0, 1.0 / k, 0.0)
    for _ in range(1000):
        over = w > cap + 1e-12
        if not over.any():
            return w
        excess = (w[over] - cap).sum()
        w[over] = cap
        free = ~over & (w > 0)
        if not free.any():
            return np.where(w > 0, 1.0 / k, 0.0)
        w[free] += excess * w[free] / w[free].sum()
    return w


def _solve(prob):
    for solver in ("CLARABEL", "SCS", "OSQP"):
        try:
            prob.solve(solver=solver)
            if prob.status in ("optimal", "optimal_inaccurate"):
                return
        except cp.error.SolverError:
            continue
    raise RuntimeError(f"all solvers failed (status={prob.status})")


def min_variance(Sigma, cap=CAP):
    n = Sigma.shape[0]
    w = cp.Variable(n)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(Sigma))),
                      [cp.sum(w) == 1, w >= 0, w <= cap])
    _solve(prob)
    return _apply_cap(np.asarray(w.value).ravel(), cap)


def max_diversification(Sigma, cap=CAP):
    """Choueifaty-Coignard via the synthetic-asset QP:
    min y'Sigma y  s.t.  sigma'y = 1, y >= 0; then w = y / sum(y)."""
    n = Sigma.shape[0]
    sigma = np.sqrt(np.diag(Sigma))
    y = cp.Variable(n)
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, cp.psd_wrap(Sigma))),
                      [sigma @ y == 1, y >= 0])
    _solve(prob)
    w = np.asarray(y.value).ravel()
    return _apply_cap(w / w.sum(), cap)


def risk_parity(Sigma, cap=CAP):
    """Equal risk contribution (Spinu log-barrier)."""
    n = Sigma.shape[0]
    w = cp.Variable(n, pos=True)
    prob = cp.Problem(cp.Minimize(0.5 * cp.quad_form(w, cp.psd_wrap(Sigma))
                                  - cp.sum(cp.log(w)) / n))
    _solve(prob)
    wv = np.asarray(w.value).ravel()
    return _apply_cap(wv / wv.sum(), cap)


OPTIMIZERS = {"min_variance": min_variance,
              "max_diversification": max_diversification,
              "risk_parity": risk_parity}


# --------------------------------------------- rolling OOS backtest ------------
def run_backtest(dataset, cov_estimator, optimizer, *, lookback=LOOKBACK, cap=CAP):
    """No look-ahead: weights at month i use excess returns through i (trailing
    `lookback`), realised on month i+1's total returns. Monthly rebalance,
    long-only, fully invested. Returns gross returns + per-rebalance one-way
    turnover; the engine's net (C1) is exactly gross - turnover*cost, weights
    unchanged — so nets derive bit-identically from one gross run."""
    rets, rf, mkt = dataset["returns"], dataset["rf"], dataset["mkt_excess"]
    excess = rets.sub(rf, axis=0)
    idx = rets.index
    w_prev, port, turns, ret_dates = None, [], [], []
    for i in range(lookback - 1, len(idx) - 1):
        win = excess.iloc[i - lookback + 1:i + 1].dropna(axis=1)   # full-history assets
        if win.shape[1] < 2:
            continue
        Sigma = cov_estimator(win.values, mkt.iloc[i - lookback + 1:i + 1].values)
        w = pd.Series(optimizer(Sigma, cap), index=win.columns)
        nxt = rets.iloc[i + 1].reindex(w.index).fillna(0.0)        # realise on i+1
        gross = float((w * nxt).sum())
        if w_prev is None:
            turnover = 1.0
        else:
            allk = w.index.union(w_prev.index)
            turnover = float((w.reindex(allk, fill_value=0)
                              - w_prev.reindex(allk, fill_value=0)).abs().sum() / 2)
        port.append(gross)
        turns.append(turnover)
        ret_dates.append(idx[i + 1])
        w_prev = w
    di = pd.DatetimeIndex(ret_dates)
    return pd.Series(port, index=di, name="gross"), pd.Series(turns, index=di, name="turnover")


# ----------------------------------------------------------- metrics -----------
@dataclass
class Metrics:
    n_months: int
    ann_return: float      # geometric (CAGR) on total returns
    ann_vol: float
    sharpe: float          # annualised, on excess returns
    t_stat: float          # t-stat of the mean monthly excess return
    max_drawdown: float
    skew: float            # of monthly excess returns (for DSR)
    kurtosis: float        # normal = 3 (for DSR)
    ann_excess: float      # arithmetic mean monthly excess x 12


def compute(returns, rf):
    r = returns.dropna()
    e = (r - rf.reindex(r.index)).dropna()
    sd = e.std(ddof=1)
    wealth = (1 + r).cumprod()
    return Metrics(
        n_months=len(r),
        ann_return=float((1 + r).prod() ** (PPY / len(r)) - 1),
        ann_vol=float(r.std(ddof=1) * np.sqrt(PPY)),
        sharpe=float(e.mean() / sd * np.sqrt(PPY)),
        t_stat=float(e.mean() / sd * np.sqrt(len(e))),
        max_drawdown=float((wealth / wealth.cummax() - 1).min()),
        skew=float(stats.skew(e, bias=False)),
        kurtosis=float(stats.kurtosis(e, fisher=False, bias=False)),
        ann_excess=float(e.mean() * PPY),
    )


# ------------------------------- C2 deflation gates (G1 + G2) ------------------
_GAMMA_E = 0.5772156649  # Euler-Mascheroni


def deflated_sharpe(sr_periodic, T, skew, kurt, sr_trials, n_trials=9, dsr_pass=0.95):
    """Bailey-Lopez de Prado Deflated Sharpe Ratio (monthly Sharpes, best-of-N)."""
    sr_trials = np.asarray(sr_trials, dtype=float)
    N = max(int(n_trials), 2)
    var_sr = float(np.var(sr_trials, ddof=1))
    sr_thresh = np.sqrt(var_sr) * ((1 - _GAMMA_E) * norm.ppf(1 - 1.0 / N)
                                   + _GAMMA_E * norm.ppf(1 - 1.0 / (N * np.e)))
    denom = np.sqrt(1 - skew * sr_periodic + (kurt - 1) / 4 * sr_periodic ** 2)
    dsr = float(norm.cdf((sr_periodic - sr_thresh) * np.sqrt(T - 1) / denom))
    return dsr, dsr > dsr_pass


def multiple_testing(tstats, alpha=0.05):
    """BHY (FDR, arbitrary dependence) over the 9 trial t-stats + the
    Harvey-Liu-Zhu t>=3 floor on the best config."""
    keys = list(tstats)
    t = np.array([tstats[k] for k in keys], dtype=float)
    p = 2 * (1 - norm.cdf(np.abs(t)))
    M = len(p)
    order = np.argsort(p)
    cM = np.sum(1.0 / np.arange(1, M + 1))
    bhy = np.zeros(M, bool)
    for rank in range(M - 1, -1, -1):
        i = order[rank]
        if p[i] <= (rank + 1) / (M * cM) * alpha:
            bhy[order[: rank + 1]] = True
            break
    best = int(np.argmax(t))
    hlz_floor = bool(t[best] >= 3.0)
    return {"survivors_bhy": [keys[i] for i in range(M) if bhy[i]],
            "hlz_t3_floor_met": hlz_floor,
            "focal_pass": bool(bhy[best] and hlz_floor)}


# ------------------------------------------------------------------ main -------
def main():
    t0 = time.time()
    print("Pulling the Ken French Data Library (49 industries + FF factors)…")
    ds = load_dataset()
    print(f"  {ds['returns'].shape[1]} assets x {len(ds['returns'])} months "
          f"({ds['returns'].index.min().date()} -> {ds['returns'].index.max().date()})")

    print("Running the 3x3 grid (9 configs x 588 monthly CVXPY solves)…")
    grid = {}
    for cov_name, cov_fn in ESTIMATORS.items():
        for opt_name, opt_fn in OPTIMIZERS.items():
            gross, turns = run_backtest(ds, cov_fn, opt_fn)
            net = gross - turns * COST_BPS / 1e4          # exactly the engine's C1
            grid[(opt_name, cov_name)] = {
                "gross": compute(gross, ds["rf"]), "net": compute(net, ds["rf"]),
                "net_ret": net, "turnover": float(turns.mean())}
            print(f"  {opt_name:20s}+ {cov_name:22s}done ({time.time()-t0:4.0f}s)")

    n = grid[FOCAL]["gross"].n_months
    print(f"\n=== C0 grid — gross annualised Sharpe (3 cov x 3 optimizers), "
          f"n={n}, OOS 1975-2023 ===")
    print(f"{'cov \\ optimizer':>20s}{'min-var':>10s}{'max-div':>10s}{'risk-par':>10s}")
    note = {"constant_correlation": "   <- max-div = the headline (claim 1.04)",
            "sample_shrinkage": "   <- max-div = the best variant (claim 1.34)"}
    for cov in ESTIMATORS:
        row = "".join(f"{grid[(o, cov)]['gross'].sharpe:10.3f}" for o in OPTIMIZERS)
        print(f"{cov:>20s}{row}{note.get(cov, '')}")

    fm, bm = grid[FOCAL]["net"], grid[BEST]["net"]
    g1 = multiple_testing({"+".join(k): v["net"].t_stat for k, v in grid.items()})
    sr_mo = {k: v["net"].sharpe / np.sqrt(PPY) for k, v in grid.items()}
    dsr, g2_pass = deflated_sharpe(sr_mo[FOCAL], fm.n_months, fm.skew, fm.kurtosis,
                                   np.array(list(sr_mo.values())))

    print("\n=== C2 deflation (net of 10 bps/side; best-of-9 selection) ===")
    print(f"G1 multiple-testing: BHY survivors {len(g1['survivors_bhy'])}/9, "
          f"Harvey-Liu-Zhu t>=3 floor met={g1['hlz_t3_floor_met']} "
          f"-> focal pass={g1['focal_pass']}")
    print(f"G2 deflated Sharpe: DSR={dsr:.2f} (pass > 0.95) -> {g2_pass}")

    print("\n=== Money-shot: claimed -> measured (net of 10 bps/side) ===")
    print(f"{'':40s}{'claimed':>9s}{'measured':>10s}")
    print(f"{'Headline Sharpe (max-div + const-corr)':40s}{1.04:>9.2f}"
          f"{fm.sharpe:>10.3f}   (t={fm.t_stat:.2f})")
    print(f"{'Best variant (max-div + shrinkage)':40s}{1.34:>9.2f}{bm.sharpe:>10.3f}")
    print(f"{'Excess return, %/yr':40s}{'+41.8':>9s}{fm.ann_excess*100:>+10.1f}")
    print(f"\nVERDICT: OVERSTATED — measured {fm.sharpe:.3f} is ~{fm.sharpe/1.04*100:.0f}% "
          f"of the claimed 1.04; statistically real, magnitude ~2x overstated,")
    print("and what remains is market beta, not alpha (FF3 alpha ~0, R^2=0.95 — report C7).")
    print(f"Full C0-C7 gate suite -> https://market-graphs.com/praxis/papers/363172 "
          f"({time.time()-t0:.0f}s)")
    return {"grid": grid, "ds": ds}

In [ ]:
res = main()

In [ ]:
import matplotlib.pyplot as plt

focal_net = res["grid"][FOCAL]["net_ret"]
mkt_tot = (res["ds"]["mkt_excess"] + res["ds"]["rf"]).reindex(focal_net.index)
mkt_sr = float(res["ds"]["mkt_excess"].reindex(focal_net.index).mean()
               / res["ds"]["mkt_excess"].reindex(focal_net.index).std(ddof=1) * np.sqrt(12))
eq_s, eq_m = (1 + focal_net).cumprod(), (1 + mkt_tot).cumprod()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eq_s.index, eq_s.values, color="#A93226", lw=1.4,
        label=f"Max-div + constant-correlation, net (measured SR "
              f"{res['grid'][FOCAL]['net'].sharpe:.2f})")
ax.plot(eq_m.index, eq_m.values, color="#051C5B", lw=1.4, alpha=0.85,
        label=f"US market total return (SR {mkt_sr:.2f})")
ax.set_yscale("log")
ax.set_title("The claimed-1.04 strategy is the market in disguise — growth of $1, 1975-2023 (log)")
ax.set_ylabel("Growth of $1")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## What you just reproduced
- **The full 3×3 grid** — all nine gross Sharpes land on the published C0 table (0.595–0.664; the report's money-shot table shows the focal 0.595 and best 0.628 at 2 dp as 0.60 / 0.63).
- **Headline (max-div + constant-correlation): claimed 1.04 → measured 0.595** (t=4.16); best variant claimed 1.34 → measured 0.628; claimed **+41.8%/yr** excess → measured **+9.6%/yr**.
- **C2: statistically real** — 9/9 configs survive BHY FDR, the Harvey-Liu-Zhu t≥3 floor is met, DSR = 1.00. The strategy earns *something*; it just isn't what the paper claims — the chart shows what it actually is: the market (correlation 0.964, β_mkt 0.99).

## The honest caveats
- The paper's universe is **undisclosed** ("1000 risky US stocks, CRSP" — no selection rule anywhere in the text; a constant 1,000 names over 53 years with no delisting handling → survivorship-suspected). We validate on the survivorship-clean 49-industry panel; the substitution is also the correct survivorship control, but a clean *originality* verdict would need the real CRSP universe (see the report's C7 caveat).
- Weight cap (0.10), shrinkage target (Ledoit-Wolf) and the 60-month lookback are **estimated** — the paper doesn't disclose them.
- Costs are immaterial here (~0.007/mo one-way turnover on industry portfolios): gross ≈ net.
- Ken French **revises history** slightly over time, so a grid cell can drift ±0.001 against the published table (on the report's data vintage every cell matches it exactly).

**This is Praxis: not a strategy to trade — a reproducible audit of whether the alpha is real.**
